# Harmony360 Core Reconciliation — Snapshot + Ledger v1

This notebook verifies the reconciled state/evidence layer:

- `Harmony360Snapshot`
- `HAR360LedgerManager`
- `h360_sync`

Historical lineage:
- `Harmony360Snapshot` had implemented save/load/export/import behavior in the recovered v13 lineage.
- `HAR360LedgerManager` existed as a named architectural stub.
- `h360_sync` historically wrote `.har360` snapshots after decorated operations.

Reconciliation goal:
**operation → verified snapshot → artifact hash → append-only ledger entry → hash-chain verification → evidence**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!python -m pip install -q numpy pytest cryptography
!rm -rf /content/Harmony360
!git clone -q https://github.com/rdntmsn/Harmony360.git /content/Harmony360
%cd /content/Harmony360/reconciliation/core_runtime_v1
!python -m pip install -q -e .


In [ ]:
import json, subprocess, sys, platform, hashlib
from pathlib import Path
import numpy, pytest, cryptography

commit = subprocess.check_output(["git","rev-parse","HEAD"], text=True).strip()
print("Git commit:", commit)
print("Python:", sys.version.split()[0])
print("NumPy:", numpy.__version__)
print("pytest:", pytest.__version__)
print("cryptography:", cryptography.__version__)


In [ ]:
# Run the COMPLETE runtime-spine suite.
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

assert result.returncode == 0, "Runtime-spine test suite failed."


In [ ]:
from harmony360 import Harmony360Snapshot, HAR360LedgerManager, h360_sync

ROOT = Path("/content/drive/MyDrive/HAR360_MASTER/reconciliation/snapshot_ledger")
RUN_DIR = ROOT / "runs"
SNAPSHOT_DIR = RUN_DIR / "snapshots"
LEDGER_PATH = RUN_DIR / "HAR360_LEDGER.jsonl"

ROOT.mkdir(parents=True, exist_ok=True)
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

@h360_sync(
    SNAPSHOT_DIR,
    LEDGER_PATH,
    snapshot_type="reconciliation_probe",
    notes="Snapshot+Ledger reconciliation verification",
    lineage="recovered snapshot + historical h360_sync + ledger stub -> canonical runtime v1.1",
)
def reconciliation_probe():
    return {
        "capability": "snapshot_ledger",
        "value": 360,
        "status": "probe_complete",
    }

probe = reconciliation_probe()
ledger = HAR360LedgerManager(LEDGER_PATH)
verification = ledger.verify_chain()

print("Probe:", probe)
print("Ledger verification:", json.dumps(verification, indent=2))
assert verification["valid"] is True


In [ ]:
# Build sealed evidence record only after full tests and ledger verification pass.
from datetime import datetime, timezone

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

tracked_files = [
    Path("harmony360/core.py"),
    Path("harmony360/constants.py"),
    Path("harmony360/serialization.py"),
    Path("harmony360/evidence.py"),
    Path("harmony360/snapshot.py"),
    Path("harmony360/ledger.py"),
    Path("harmony360/sync.py"),
    Path("harmony360/security/crypto_vault.py"),
    Path("tests/test_core.py"),
    Path("tests/test_serialization.py"),
    Path("tests/test_crypto_vault.py"),
    Path("tests/test_snapshot_ledger.py"),
    Path("pyproject.toml"),
]

file_hashes = {str(p): sha256_file(p) for p in tracked_files if p.exists()}

evidence = {
    "capability": "Harmony360 Snapshot + Ledger Reconciliation v1",
    "classification": "CORE_RUNTIME_RECONCILIATION",
    "status": "PASS",
    "sealed_utc": datetime.now(timezone.utc).isoformat(),
    "git": {
        "repository": "rdntmsn/Harmony360",
        "branch": "main",
        "commit_sha": commit,
        "root_path": "reconciliation/core_runtime_v1",
    },
    "runtime": {
        "python": sys.version.split()[0],
        "numpy": numpy.__version__,
        "pytest": pytest.__version__,
        "cryptography": cryptography.__version__,
        "platform": platform.platform(),
    },
    "tests": {
        "command": "python -m pytest -q",
        "returncode": result.returncode,
        "stdout": result.stdout.strip(),
    },
    "snapshot_ledger_probe": {
        "ledger_path": str(LEDGER_PATH),
        "verification": verification,
        "snapshot_count": len(list(SNAPSHOT_DIR.glob("*.har360.json"))),
    },
    "source_sha256": file_hashes,
    "scope_of_conclusion": (
        "Verifies tested snapshot serialization/restoration, snapshot tamper detection, "
        "append-only ledger hash chaining, ledger tamper detection, and h360_sync linkage "
        "within the tested runtime. This is an engineering/state-integrity result and does "
        "not by itself establish empirical scientific claims."
    ),
}

canonical = json.dumps(evidence, sort_keys=True, separators=(",", ":")).encode()
evidence["record_sha256"] = hashlib.sha256(canonical).hexdigest()

evidence_path = ROOT / "SNAPSHOT_LEDGER_RECONCILIATION_v1_SEALED.har360.json"
evidence_path.write_text(json.dumps(evidence, indent=2, sort_keys=True), encoding="utf-8")

print("Evidence:", evidence_path)
print("Record SHA256:", evidence["record_sha256"])


## Promotion boundary

A PASS here means the state/evidence mechanics behave as tested:
- state round-trip works,
- tampering is detected,
- ledger entries are hash chained,
- altered history is detected,
- decorated operations produce linked snapshots and ledger entries.

It does **not** mean scientific claims stored in a snapshot are true. The ledger preserves evidence and lineage; it does not manufacture truth.
